# Persona probe — does "you are a senior engineer" change the model? (inference-time A/B)

**Runtime: any GPU** (A100 ~30-45 min, T4 works but ~2h). Dev-slice only —
no training, no exam, safe to run IN PARALLEL with notebook 18 (reads
`sft_v2_s3407_ep2`, writes only `dev_eval_persona_*` files).

**The question:** if we tell the SFT v2 model it is a senior engineer, does
its bug-fixing behaviour change? Three arms, same 60 new-dev bugs, k=8:

1. **standard** — the exact dev prompt (control; must reproduce ~73.5/91.7,
   else the probe is void)
2. **persona_system** — persona as a system message (this *replaces* Qwen's
   built-in default system message)
3. **persona_prefix** — persona prepended inside the user message

**Registered predictions (BEFORE running — S2.31):**
- All three arms within the ±3-4 dev noise band of each other.
- If anything, persona_prefix leans *negative* — it perturbs the trained
  input format more (the 13b format-toxicity lesson).
- Note the ceiling on this idea: the exam prompt is FROZEN, so even a dev win
  could never touch the headline number. This is a behaviour/robustness probe.
  A *training-time* persona twin (persona in the SFT data) is notebook 20,
  built only if this probe shows a real effect.

In [ ]:
# 1) Setup
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random
PHASE8 = '/content/drive/MyDrive/rl-post-training/phase8'
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
v1_bugs = json.load(open(f'{PHASE8}/data_v1_bugs.json'))
dev_new = [b for b in v1_bugs if b['split'] == 'dev' and b.get('gen') == 'deepseek_v1']
dev_new_sample = random.Random(3407).sample(dev_new, 60)   # the standard 60
print(len(dev_new_sample), 'dev bugs')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# 2) Grading + the three prompt arms
import subprocess, tempfile, torch
from concurrent.futures import ThreadPoolExecutor
from unsloth import FastLanguageModel

def passes(code, rec, timeout=6):
    prog = '\n'.join(list(rec['test_imports']) + [code] + list(rec['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

PERSONA = ('You are a senior software engineer with 15 years of experience '
           'debugging production Python. You fix bugs with minimal, precise edits.')

def msgs_standard(b):
    return [{'role': 'user', 'content': build_repair_prompt(b['buggy'], b['test_list'])}]

def msgs_system(b):
    return [{'role': 'system', 'content': PERSONA},
            {'role': 'user', 'content': build_repair_prompt(b['buggy'], b['test_list'])}]

def msgs_prefix(b):
    return [{'role': 'user',
             'content': PERSONA + '\n\n' + build_repair_prompt(b['buggy'], b['test_list'])}]

ARMS = {'standard': msgs_standard,
        'persona_system': msgs_system,
        'persona_prefix': msgs_prefix}

def arm_eval(model, tok, arm, msgs_fn, tag, k=8):
    per_bug = []
    for i, b in enumerate(dev_new_sample):
        text = tok.apply_chat_template(msgs_fn(b), tokenize=False,
                                       add_generation_prompt=True)
        enc = tok([text], return_tensors='pt', padding=True,
                  padding_side='left').to('cuda')
        with torch.no_grad():
            out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                                 num_return_sequences=k, max_new_tokens=512,
                                 pad_token_id=tok.eos_token_id)
        gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:],
                               skip_special_tokens=True)
        codes = [extract_code(t) for t in gen]
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
        if (i + 1) % 20 == 0:
            print(f'  {arm}: {i+1}/60')
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    pk = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    json.dump({'tag': tag, 'arm': arm, 'pass@1': p1, 'pass@8': pk, 'per_bug': per_bug},
              open(f'{PHASE8}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@8={pk*100:.1f}%")

# one sample echo so the rendered prompts are on the record
_tok_check = None
print('persona text:', PERSONA)

In [ ]:
# 3) THE PROBE — SFT v2 s3407, three arms (checkpointed per arm)
model, tok = FastLanguageModel.from_pretrained(
    f'{PHASE8}/sft_v2_s3407_ep2', max_seq_length=1280,
    load_in_4bit=True, dtype=None)
FastLanguageModel.for_inference(model)

# show exactly what each arm feeds the model (first 300 chars)
b0 = dev_new_sample[0]
for arm, fn in ARMS.items():
    text = tok.apply_chat_template(fn(b0), tokenize=False, add_generation_prompt=True)
    print(f'--- {arm} ---')
    print(text[:300].replace(chr(10), ' / '), '...')

for arm, fn in ARMS.items():
    tag = f'persona_{arm}_sftv2_s3407'
    if os.path.exists(f'{PHASE8}/dev_eval_{tag}.json'):
        print(arm, 'already done, skipping'); continue
    arm_eval(model, tok, arm, fn, tag)

In [ ]:
# 4) Optional: same three arms on the UNTRAINED base model.
# Contrast: has SFT overwritten whatever persona-sensitivity the base had?
# Doubles the runtime — flip to True only if you want the contrast.
RUN_BASE = False
if RUN_BASE:
    import gc
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    model, tok = FastLanguageModel.from_pretrained(
        'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1280,
        load_in_4bit=True, dtype=None)
    FastLanguageModel.for_inference(model)
    for arm, fn in ARMS.items():
        tag = f'persona_{arm}_base'
        if os.path.exists(f'{PHASE8}/dev_eval_{tag}.json'):
            print(arm, 'already done, skipping'); continue
        arm_eval(model, tok, arm, fn, tag)
else:
    print('base contrast skipped (RUN_BASE=False)')

In [ ]:
# 5) VERDICT vs the registered predictions
def load(tag):
    p = f'{PHASE8}/dev_eval_{tag}.json'
    return json.load(open(p)) if os.path.exists(p) else None
print(f"{'arm (SFT v2 s3407, 60 new-dev)':<32} pass@1   pass@8")
print(f"{'(nb 13 reference: standard)':<32}  73.5%    91.7%")
base_row = None
for arm in ARMS:
    r = load(f'persona_{arm}_sftv2_s3407')
    if r:
        print(f"{arm:<32} {r['pass@1']*100:6.1f}%  {r['pass@8']*100:6.1f}%")
    else:
        print(f'{arm:<32} (not run)')
for arm in ARMS:
    r = load(f'persona_{arm}_base')
    if r:
        if base_row is None:
            print('\n-- base model contrast --'); base_row = True
        print(f"{'base ' + arm:<32} {r['pass@1']*100:6.1f}%  {r['pass@8']*100:6.1f}%")
print('\nRegistered (S2.31): all arms within +/-3-4 of each other; standard must')
print('reproduce ~73.5/91.7 or the probe is void; persona_prefix leans negative.')
print('Reminder: the exam prompt is FROZEN — a dev win here cannot change the')
print('headline. Training-time persona twin = notebook 20, only if an effect shows.')
print('\nBring the whole printout back.')

## Bring back to the session
The **verdict table** (and the three rendered-prompt previews from cell 3).
Runs in parallel with notebook 18 — different Colab session, no shared writes.